In [0]:
#Read Required Tables
drivers = spark.table("workspace.logistics_project.drivers")
loads = spark.table("workspace.logistics_project.loads")
trips = spark.table("workspace.logistics_project.trips")
delivery_events = spark.table("workspace.logistics_project.delivery_events")

In [0]:
#Join Tables
driver_perf = (
    trips
    .join(loads, "load_id")
    .join(drivers, "driver_id")
    .join(delivery_events, "trip_id", "left")
)

In [0]:
#Create driver_metrics
from pyspark.sql.functions import *

driver_metrics = (
    driver_perf
    .groupBy("driver_id","first_name","last_name")
    .agg(
        round(avg("average_mpg"),2).alias("avg_mpg"),
        round(
            avg(
                when(col("on_time_flag")==True,1)
                .otherwise(0)
            ) * 100,2
        ).alias("on_time_rate_pct"),
        round(sum("revenue"),2).alias("total_revenue"),
        sum("actual_distance_miles").alias("total_miles")
    )
)

driver_metrics = driver_metrics.withColumn(
    "revenue_per_mile",
    round(col("total_revenue")/col("total_miles"),2)
)

In [0]:
#Display Results
display(driver_metrics)

driver_id,first_name,last_name,avg_mpg,on_time_rate_pct,total_revenue,total_miles,revenue_per_mile
DRV00071,Jessica,Johnson,6.48,55.91,3997206.48,1898920,2.1
DRV00125,David,Smith,6.5,56.39,4189201.72,1952260,2.15
DRV00050,Robert,Johnson,6.48,56.31,3954636.1,1835482,2.15
DRV00104,Richard,Hernandez,6.5,56.8,4028058.24,1869338,2.15
DRV00069,Patricia,Taylor,6.5,54.58,4369927.8,2040182,2.14
DRV00012,Robert,Moore,6.52,54.69,3499731.32,1672374,2.09
DRV00098,David,Anderson,6.55,53.89,4193424.52,1955754,2.14
DRV00146,Jessica,Gonzalez,6.49,54.15,4185215.78,1930904,2.17
DRV00097,Thomas,Rodriguez,6.48,54.69,3950207.28,1867362,2.12
DRV00053,John,Martin,6.49,56.99,3881691.36,1814452,2.14


In [0]:
display(driver_metrics)

driver_id,first_name,last_name,avg_mpg,on_time_rate_pct,total_revenue,total_miles,revenue_per_mile
DRV00071,Jessica,Johnson,6.48,55.91,3997206.48,1898920,2.1
DRV00125,David,Smith,6.5,56.39,4189201.72,1952260,2.15
DRV00050,Robert,Johnson,6.48,56.31,3954636.1,1835482,2.15
DRV00104,Richard,Hernandez,6.5,56.8,4028058.24,1869338,2.15
DRV00069,Patricia,Taylor,6.5,54.58,4369927.8,2040182,2.14
DRV00012,Robert,Moore,6.52,54.69,3499731.32,1672374,2.09
DRV00098,David,Anderson,6.55,53.89,4193424.52,1955754,2.14
DRV00146,Jessica,Gonzalez,6.49,54.15,4185215.78,1930904,2.17
DRV00097,Thomas,Rodriguez,6.48,54.69,3950207.28,1867362,2.12
DRV00053,John,Martin,6.49,56.99,3881691.36,1814452,2.14


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.logistics_gold
""")

DataFrame[]

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show()

+--------------+------------------+-----------+
|      database|         tableName|isTemporary|
+--------------+------------------+-----------+
|logistics_gold|driver_performance|      false|
+--------------+------------------+-----------+



In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.driver_performance"
)

gold_table.alias("target").merge(
    driver_metrics.alias("source"),
    "target.driver_id = source.driver_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("workspace.logistics_gold.driver_performance")
)

driver_id,first_name,last_name,avg_mpg,on_time_rate_pct,total_revenue,total_miles,revenue_per_mile
DRV00071,Jessica,Johnson,6.48,55.91,3997206.48,1898920,2.1
DRV00125,David,Smith,6.5,56.39,4189201.72,1952260,2.15
DRV00050,Robert,Johnson,6.48,56.31,3954636.1,1835482,2.15
DRV00104,Richard,Hernandez,6.5,56.8,4028058.24,1869338,2.15
DRV00069,Patricia,Taylor,6.5,54.58,4369927.8,2040182,2.14
DRV00012,Robert,Moore,6.52,54.69,3499731.32,1672374,2.09
DRV00098,David,Anderson,6.55,53.89,4193424.52,1955754,2.14
DRV00146,Jessica,Gonzalez,6.49,54.15,4185215.78,1930904,2.17
DRV00097,Thomas,Rodriguez,6.48,54.69,3950207.28,1867362,2.12
DRV00053,John,Martin,6.49,56.99,3881691.36,1814452,2.14


In [0]:
driver_metrics.show(5)

+---------+----------+---------+-------+----------------+-------------+-----------+----------------+
|driver_id|first_name|last_name|avg_mpg|on_time_rate_pct|total_revenue|total_miles|revenue_per_mile|
+---------+----------+---------+-------+----------------+-------------+-----------+----------------+
| DRV00071|   Jessica|  Johnson|   6.48|           55.91|   3997206.48|    1898920|             2.1|
| DRV00125|     David|    Smith|    6.5|           56.39|   4189201.72|    1952260|            2.15|
| DRV00050|    Robert|  Johnson|   6.48|           56.31|    3954636.1|    1835482|            2.15|
| DRV00104|   Richard|Hernandez|    6.5|            56.8|   4028058.24|    1869338|            2.15|
| DRV00069|  Patricia|   Taylor|    6.5|           54.58|    4369927.8|    2040182|            2.14|
+---------+----------+---------+-------+----------------+-------------+-----------+----------------+
only showing top 5 rows


In [0]:
#Save the Result to Gold Layer Using MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.driver_performance"
)

gold_table.alias("target").merge(
    driver_metrics.alias("source"),
    "target.driver_id = source.driver_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]